# Pipeline de Clusterização de Clientes - Olist
Este notebook executa o pipeline completo de segmentação de clientes utilizando PCA para redução de dimensionalidade, seguido pelos algoritmos K-Means e GMM.

In [1]:
# Configuração para recarregar os arquivos .py automaticamente caso você os altere
%load_ext autoreload
%autoreload 2

import pandas as pd
import numpy as np

# Importando os seus módulos personalizados
from src.processamento import carregar_e_agregar_dados, tratar_e_padronizar
from src.dimensionalidade import aplicar_pca, plotar_variancia_pca
from src.modelagem import rodar_kmeans, rodar_gmm, comparar_modelos, analisar_perfis

## Passo 1: Consolidação dos Dados
Carregando os arquivos brutos da pasta `dados/` e agregando as informações por cliente único (`customer_unique_id`).

In [2]:
caminho_dos_dados = 'dados/'
df_clientes = carregar_e_agregar_dados(caminho_dos_dados)

print(f"Total de clientes processados: {df_clientes.shape[0]}")
df_clientes.head()

Total de clientes processados: 94990


,customer_unique_id,recencia,frequencia,monetario,parcelas_medias,frete_total,review_medio
0,0000366f3b9a7992bf8c76cfdf3221e2,116,1,141.90,8.0,12.00,5.0
1,0000b849f77a49e4a4ce2b2a4ca5be3f,119,1,27.19,1.0,8.29,4.0
2,0000f46a3911fa3c0805444483337064,542,1,86.22,8.0,17.22,3.0
3,0000f6ccb0745a6a4b88665a16c9f078,326,1,43.62,4.0,17.63,4.0
4,0004aac84e0df4da2b147fca70cf8255,293,1,196.89,6.0,16.89,5.0


## Passo 2: Pré-processamento e Padronização
Definição das features que alimentarão o modelo, aplicação de log para corrigir assimetrias e padronização com `StandardScaler`.

In [ ]:
features = ['recencia', 'frequencia', 'monetario', 'parcelas_medias', 'frete_total', 'review_medio']

# Esta função retorna o array pronto para os modelos
dados_padronizados = tratar_e_padronizar(df_clientes, features)
print(f"Formato dos dados padronizados: {dados_padronizados.shape}")

## Passo 3: Análise de Dimensionalidade (PCA)
Ajustando o PCA para entender quanta variância cada componente consegue explicar.

In [ ]:
pca_modelo = aplicar_pca(dados_padronizados)

# Renderiza o gráfico de variância acumulada para tomarmos a decisão do corte
plotar_variancia_pca(pca_modelo)

### Redução para os Componentes Selecionados
Com base no gráfico acima, escolha o número de componentes (ex: 3 componentes) para transformar nossos dados.

In [ ]:
n_componentes = 3  # Altere este número baseado no seu gráfico de variância

# Filtrando o array para conter apenas os N primeiros componentes escolhidos
dados_pca = pca_modelo.transform(dados_padronizados)[:, :n_componentes]
print(f"Dados prontos para clusterização. Novo formato: {dados_pca.shape}")

## Passo 4: Agrupamento com K-Means
Execução do loop para avaliação de hiperparâmetros (Inércia e Silhueta). Ajuste do modelo final com o `k_escolhido`.

In [ ]:
# O código vai plotar o Cotovelo e a Silhueta automaticamente
# Defina o k_escolhido após analisar os gráficos gerados por essa função
kmeans_modelo, labels_kmeans = rodar_kmeans(dados_pca, k_max=10, k_escolhido=4)

# Salvando o resultado no nosso dataframe principal
df_clientes['cluster_kmeans'] = labels_kmeans

## Passo 5: Agrupamento com GMM (Gaussian Mixture Models)
Avaliação dos critérios de informação BIC e AIC para escolher o número de componentes probabilísticos.

In [ ]:
# O código vai plotar as curvas de BIC e AIC automaticamente
gmm_modelo, labels_gmm = rodar_gmm(dados_pca, n_max=10, n_escolhido=4)

# Salvando os rótulos do GMM e a probabilidade de pertencer ao cluster principal
df_clientes['cluster_gmm'] = labels_gmm
df_clientes['prob_cluster'] = gmm_modelo.predict_proba(dados_pca).max(axis=1)

## Passo 6: Comparação Estatística entre K-Means e GMM
Análise de cruzamento de grupos (Matriz de Contingência), ARI e NMI para checar a concordância dos dois modelos.

In [ ]:
comparar_modelos(df_clientes['cluster_kmeans'], df_clientes['cluster_gmm'])

## Passo 7: Perfilamento de Negócio
Análise das médias reais de faturamento, recência, frete e avaliações para criar as personas de cada cluster.

In [ ]:
print("====== PERFIL DOS CLUSTERS - K-MEANS ======")
perfis_kmeans = analisar_perfis(df_clientes, agrupado_por='cluster_kmeans')
display(perfis_kmeans)

print("\n====== PERFIL DOS CLUSTERS - GMM ======")
perfis_gmm = analisar_perfis(df_clientes, agrupado_por='cluster_gmm')
display(perfis_gmm)